# 🏥 RxHCC FWA Investigation Agent
### Kaggle 5-Day AI Agents Intensive Vibe Coding — Agents for Good Track

**Author:** Gyver (sechan9999)  
**Live Demo:** https://rxhcc-app.vercel.app/  
**GitHub:** https://github.com/sechan9999/rxhcc_agent

---

## What This Does

Medicare fraud costs U.S. taxpayers **\$60–100 billion per year**. Traditional rule-based systems catch fraud *after* payment — forcing costly "pay and chase" recoveries.

This notebook demonstrates an **autonomous multi-agent AI system** that investigates Medicare Part D drug claims *before payment is released*, using:
- **Google ADK** (Agent Development Kit) for multi-agent orchestration
- **Gemini 2.0 Flash** for reasoning at each agent step
- **5 specialized tools** for ICD-10 validation, drug combo detection, provider risk scoring, and report generation

## Agent Architecture

```
User submits claim
      │
      ▼
fwa_orchestrator  (Gemini 2.0 Flash)
  ├── claim_analyzer   → lookup_icd10_code
  ├── risk_scorer      → calculate_rxhcc_risk_score
  │                       check_drug_combination
  │                       get_provider_billing_history
  └── report_writer    → generate_fwa_report
```

## Step 1 — Install Dependencies

In [ ]:
!pip install -q google-adk google-genai streamlit python-dotenv

## Step 2 — Configure API Key

In [ ]:
import os

# Set your Gemini API key here (or use Kaggle Secrets)
# In Kaggle: Add Secret named GOOGLE_API_KEY in the notebook settings
from kaggle_secrets import UserSecretsClient
try:
    secrets = UserSecretsClient()
    os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
    print("✅ API key loaded from Kaggle Secrets")
except Exception:
    # Fallback: set manually
    os.environ["GOOGLE_API_KEY"] = "YOUR_KEY_HERE"  # <-- replace if running locally
    print("⚠️  Using hardcoded key — prefer Kaggle Secrets in production")

## Step 3 — Load the Tools

Five deterministic Python functions registered as ADK tools. Each tool represents a specific FWA detection signal.

In [ ]:
# Copy tools.py content inline so this notebook is self-contained
# (In the full repo, just: from tools import *)

from datetime import datetime
from typing import Optional

ICD10_DB = {
    "E11.9":   {"desc": "Type 2 diabetes mellitus without complications",        "severity": 1, "gender": None, "hcc": True},
    "E11.65":  {"desc": "Type 2 diabetes mellitus with hyperglycemia",           "severity": 2, "gender": None, "hcc": True},
    "I10":     {"desc": "Essential (primary) hypertension",                      "severity": 1, "gender": None, "hcc": False},
    "I50.9":   {"desc": "Heart failure, unspecified",                            "severity": 4, "gender": None, "hcc": True},
    "N18.6":   {"desc": "End-stage renal disease",                              "severity": 5, "gender": None, "hcc": True},
    "Z79.4":   {"desc": "Long-term current use of insulin",                     "severity": 1, "gender": None, "hcc": False},
    "C50.911": {"desc": "Malignant neoplasm, right female breast",              "severity": 5, "gender": "F",  "hcc": True},
    "F11.10":  {"desc": "Opioid abuse, uncomplicated",                          "severity": 3, "gender": None, "hcc": True},
    "G89.29":  {"desc": "Other chronic pain",                                   "severity": 2, "gender": None, "hcc": False},
}

NDC_DB = {
    "00406051201": {"name": "OxyContin (oxycodone) 80mg",  "schedule": "II",  "risk": "HIGH"},
    "59011049010": {"name": "Alprazolam (Xanax) 2mg",      "schedule": "IV",  "risk": "MEDIUM"},
    "00002143480": {"name": "Insulin glargine (Lantus)",   "schedule": None,  "risk": "LOW"},
    "65162010850": {"name": "Carisoprodol (Soma) 350mg",   "schedule": "IV",  "risk": "MEDIUM"},
}

PROVIDER_DB = {
    "1234567890": {
        "name": "Dr. James Rutherford MD", "specialty": "Pain Management",
        "anomaly_score": 0.91, "peer_percentile": 99,
        "flags": ["TOP_1PCT_OPIOID_PRESCRIBER", "MULTI_PATIENT_OVERLAP"],
    },
    "9876543210": {
        "name": "Sunrise Specialty Pharmacy LLC", "specialty": "Retail Pharmacy",
        "anomaly_score": 0.87, "peer_percentile": 98,
        "flags": ["DISPENSING_WITHOUT_VALID_RX", "HIGH_REFILL_RATE"],
    },
}

def lookup_icd10_code(code: str) -> dict:
    """Validate an ICD-10-CM code and return clinical details."""
    code = code.upper().strip()
    entry = ICD10_DB.get(code)
    if entry:
        return {"valid": True, "code": code, "description": entry["desc"],
                "severity": entry["severity"], "gender_restriction": entry["gender"],
                "hcc_relevant": entry["hcc"]}
    return {"valid": False, "code": code, "description": "Code not found in ICD-10-CM database"}

def get_provider_billing_history(npi: str) -> dict:
    """Retrieve provider billing statistics and risk flags."""
    if npi in PROVIDER_DB:
        return {"npi": npi, **PROVIDER_DB[npi]}
    return {"npi": npi, "name": f"Provider NPI {npi}", "anomaly_score": 0.08,
            "peer_percentile": 42, "flags": []}

def check_drug_combination(ndc_codes: list) -> dict:
    """Detect dangerous drug combinations."""
    resolved = [NDC_DB.get(n, {"name": f"Unknown ({n})", "schedule": None, "risk": "UNKNOWN"})
                for n in ndc_codes]
    has_opioid = any("oxycodone" in d["name"].lower() for d in resolved)
    has_benzo  = any("alprazolam" in d["name"].lower() for d in resolved)
    has_soma   = any("carisoprodol" in d["name"].lower() for d in resolved)
    flags = []
    if has_opioid and has_benzo:
        flags.append("OPIOID_BENZO_COMBINATION — HIGH OVERDOSE RISK (FDA Black Box)")
    if has_opioid and has_soma:
        flags.append("OPIOID_MUSCLE_RELAXANT — Documented abuse triad")
    sched_ii = [d for d in resolved if d.get("schedule") == "II"]
    if len(sched_ii) >= 2:
        flags.append("MULTIPLE_SCHEDULE_II_SUBSTANCES")
    return {
        "drugs": [d["name"] for d in resolved],
        "combination_risk": "HIGH" if flags else "LOW",
        "flags": flags,
    }

def calculate_rxhcc_risk_score(beneficiary_id, icd10_codes, ndc_codes, claim_amount, provider_npi) -> dict:
    """Calculate composite FWA fraud risk score."""
    score = 0.0
    risk_factors = []
    gender = "M" if "-M-" in beneficiary_id.upper() else ("F" if "-F-" in beneficiary_id.upper() else None)
    for code in [c.upper().strip() for c in icd10_codes]:
        info = ICD10_DB.get(code, {})
        if not info:
            score += 0.05; risk_factors.append(f"INVALID_CODE: {code}")
        elif info.get("severity", 0) >= 4:
            score += 0.10; risk_factors.append(f"HIGH_SEVERITY: {code}")
        if info.get("gender") and gender and info["gender"] != gender:
            score += 0.40; risk_factors.append(f"GENDER_MISMATCH: {code} requires {info['gender']}, patient is {gender}")
    if ndc_codes:
        dc = check_drug_combination(ndc_codes)
        if dc["combination_risk"] == "HIGH":
            score += 0.25; risk_factors.extend(dc["flags"])
    prov = get_provider_billing_history(provider_npi)
    if prov["anomaly_score"] > 0.75:
        score += 0.30; risk_factors.append(f"HIGH_RISK_PROVIDER: {prov['name']} (score {prov['anomaly_score']})"); risk_factors.extend(prov["flags"])
    if claim_amount > 5000:
        score += 0.10; risk_factors.append(f"HIGH_CLAIM_AMOUNT: ${claim_amount:,.2f}")
    score = min(round(score, 2), 1.0)
    verdict = "CLEAR" if score < 0.30 else ("FLAG_FOR_REVIEW" if score < 0.70 else "ESCALATE")
    return {"risk_score": score, "verdict": verdict, "risk_factors": risk_factors,
            "recommendation": {"CLEAR": "Approve for payment",
                               "FLAG_FOR_REVIEW": "Hold claim; request documentation",
                               "ESCALATE": "Block payment; refer to SIU"}[verdict]}

def generate_fwa_report(claim_id, risk_score, verdict, risk_factors, provider_name,
                        provider_npi, provider_anomaly_score, provider_flags,
                        drugs_prescribed, drug_combination_risk, drug_flags, recommendation) -> str:
    """Generate a structured FWA compliance report."""
    bar = "=" * 60
    emoji = {"CLEAR": "✅", "FLAG_FOR_REVIEW": "⚠️", "ESCALATE": "🚨"}.get(verdict, "❓")
    factors = "\n".join(f"  [{i+1}] {f}" for i, f in enumerate(risk_factors)) or "  None"
    return f"""{bar}
RxHCC FWA REPORT — {emoji} {verdict}
Claim: {claim_id} | Score: {risk_score:.0%} | {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}
{bar}
RECOMMENDATION: {recommendation}

RISK FACTORS ({len(risk_factors)}):
{factors}

PROVIDER: {provider_name} (NPI {provider_npi}) | Anomaly: {provider_anomaly_score:.2f}
DRUGS: {', '.join(drugs_prescribed)} | Risk: {drug_combination_risk}
{bar}"""

print("✅ Tools loaded — 5 FWA detection functions ready")

## Step 4 — Build the Multi-Agent System

Four agents are wired together using Google ADK. The orchestrator drives the pipeline; each sub-agent owns a single responsibility.

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

MODEL = "gemini-2.0-flash"

# ── Sub-Agent 1: Claim Analyzer ───────────────────────────────────────────────
claim_analyzer = Agent(
    model=MODEL,
    name="claim_analyzer",
    description="Validates ICD-10 codes and extracts structured claim fields.",
    instruction="""You are a Medicare claims validation specialist.
For each claim submitted:
1. Call lookup_icd10_code for EVERY ICD-10 code individually
2. Note validity, severity, gender restrictions
3. Infer patient gender from Beneficiary ID (-M- or -F-)
4. Output a structured summary with all findings""",
    tools=[lookup_icd10_code],
    output_key="claim_analysis",
)

# ── Sub-Agent 2: Risk Scorer ──────────────────────────────────────────────────
risk_scorer = Agent(
    model=MODEL,
    name="risk_scorer",
    description="Calculates FWA risk score using RxHCC model, drug checks, and provider history.",
    instruction="""You are an FWA risk analyst.
Using the claim data:
1. Call check_drug_combination if NDC codes are present
2. Call get_provider_billing_history with the provider NPI
3. Call calculate_rxhcc_risk_score with ALL claim fields
   (beneficiary_id, icd10_codes as list, ndc_codes as list, claim_amount as float, provider_npi)
4. Report: score, verdict, top risk factors""",
    tools=[check_drug_combination, get_provider_billing_history, calculate_rxhcc_risk_score],
    output_key="risk_assessment",
)

# ── Sub-Agent 3: Report Writer ────────────────────────────────────────────────
report_writer = Agent(
    model=MODEL,
    name="report_writer",
    description="Generates SIU-ready compliance reports from investigation findings.",
    instruction="""You are a compliance report writer.
Call generate_fwa_report with all collected findings.
Pass lists as Python lists, floats as floats.
Present the report exactly as returned.""",
    tools=[generate_fwa_report],
    output_key="final_report",
)

# ── Orchestrator ──────────────────────────────────────────────────────────────
fwa_orchestrator = Agent(
    model=MODEL,
    name="fwa_orchestrator",
    description="Root FWA investigation orchestrator. Coordinates all sub-agents.",
    instruction="""You are the RxHCC FWA Investigation Orchestrator.

For every claim:
STEP 1 → Delegate to claim_analyzer (validate codes)
STEP 2 → Delegate to risk_scorer (calculate fraud probability)
STEP 3 → Delegate to report_writer (generate compliance report)
STEP 4 → Present final verdict box:
  Claim ID | Risk Score | Verdict | Top 3 flags | Next action

Verdicts: CLEAR (<30%) | FLAG_FOR_REVIEW (30-69%) | ESCALATE (≥70%)""",
    sub_agents=[claim_analyzer, risk_scorer, report_writer],
)

print("✅ Multi-agent system built — 4 agents ready")
print("   fwa_orchestrator → [claim_analyzer, risk_scorer, report_writer]")

## Step 5 — Helper: Run an Investigation

In [ ]:
import asyncio

async def investigate_claim(claim: dict) -> str:
    """Run a full FWA investigation on a claim dict. Returns the final agent response."""
    prompt = f"""Investigate this Medicare Part D claim for Fraud, Waste & Abuse:

Claim ID:       {claim['claim_id']}
Beneficiary ID: {claim['beneficiary_id']}
ICD-10 Codes:   {', '.join(claim['icd10_codes'])}
NDC Codes:      {', '.join(claim.get('ndc_codes', [])) or 'None'}
Provider NPI:   {claim['provider_npi']}
Claim Amount:   ${claim['claim_amount']:,.2f}"""

    session_service = InMemorySessionService()
    runner = Runner(agent=fwa_orchestrator, app_name="rxhcc_demo", session_service=session_service)
    session = await session_service.create_session(app_name="rxhcc_demo", user_id="notebook")

    final = ""
    for event in runner.run(
        user_id="notebook",
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=prompt)]),
    ):
        if event.is_final_response():
            final = event.content.parts[0].text
    return final

print("✅ investigate_claim() helper ready")

## Step 6 — Demo Scenario 1: ✅ CLEAR — Clean Diabetic Claim

Routine Type 2 diabetes maintenance claim. Expect no fraud flags.

In [ ]:
claim_clean = {
    "claim_id":       "CLM-2024-001",
    "beneficiary_id": "BNF-F-99211",
    "icd10_codes":    ["E11.9", "Z79.4"],
    "ndc_codes":      ["00002143480"],   # Insulin glargine
    "provider_npi":   "1111111111",
    "claim_amount":   320.00,
}

print("🔍 Investigating clean claim...")
result_clean = asyncio.run(investigate_claim(claim_clean))
print(result_clean)

## Step 7 — Demo Scenario 2: ⚠️ FLAG — Opioid + Benzo Combination

OxyContin + Xanax co-prescribed from a flagged pharmacy. FDA Black Box warning territory.

In [ ]:
claim_suspicious = {
    "claim_id":       "CLM-2024-002",
    "beneficiary_id": "BNF-M-44831",
    "icd10_codes":    ["G89.29", "F11.10"],
    "ndc_codes":      ["00406051201", "59011049010"],  # OxyContin + Xanax
    "provider_npi":   "9876543210",                    # Flagged pharmacy
    "claim_amount":   1250.00,
}

print("🔍 Investigating suspicious claim...")
result_suspicious = asyncio.run(investigate_claim(claim_suspicious))
print(result_suspicious)

## Step 8 — Demo Scenario 3: 🚨 ESCALATE — Gender Mismatch + High-Risk Provider

Male patient billed for female breast cancer + opioid triad + watchlisted provider. Multiple ESCALATE triggers.

In [ ]:
claim_fraud = {
    "claim_id":       "CLM-2024-003",
    "beneficiary_id": "BNF-M-77542",          # MALE patient
    "icd10_codes":    ["C50.911", "E11.9"],   # Female breast cancer on male!
    "ndc_codes":      ["00406051201",          # OxyContin
                       "59011049010",          # Xanax
                       "65162010850"],         # Soma (muscle relaxant — holy trinity)
    "provider_npi":   "1234567890",            # Watchlisted pain clinic
    "claim_amount":   8400.00,
}

print("🔍 Investigating high-risk claim...")
result_fraud = asyncio.run(investigate_claim(claim_fraud))
print(result_fraud)

## Step 9 — Run All 4 Sample Claims and Summarize

In [ ]:
import json

# Load from sample_claims.json (available in the repo)
sample_claims = [
    {"claim_id": "CLM-2024-001", "beneficiary_id": "BNF-F-99211",
     "icd10_codes": ["E11.9", "Z79.4"], "ndc_codes": ["00002143480"],
     "provider_npi": "1111111111", "claim_amount": 320.00, "expected": "CLEAR"},
    {"claim_id": "CLM-2024-002", "beneficiary_id": "BNF-M-44831",
     "icd10_codes": ["G89.29", "F11.10"], "ndc_codes": ["00406051201", "59011049010"],
     "provider_npi": "9876543210", "claim_amount": 1250.00, "expected": "FLAG/ESCALATE"},
    {"claim_id": "CLM-2024-003", "beneficiary_id": "BNF-M-77542",
     "icd10_codes": ["C50.911", "E11.9"], "ndc_codes": ["00406051201", "59011049010", "65162010850"],
     "provider_npi": "1234567890", "claim_amount": 8400.00, "expected": "ESCALATE"},
]

print("Running batch evaluation on all sample claims...\n")
print(f"{'Claim ID':<15} {'Amount':>10}  {'Expected':<18} {'Result'}")
print("-" * 65)

for claim in sample_claims:
    result = asyncio.run(investigate_claim(claim))
    # Extract verdict from result text
    verdict = "ESCALATE" if "ESCALATE" in result else ("FLAG_FOR_REVIEW" if "FLAG" in result else "CLEAR")
    match = "✅" if claim["expected"].split("/")[0] in verdict else "❌"
    print(f"{claim['claim_id']:<15} ${claim['claim_amount']:>8,.0f}  {claim['expected']:<18} {match} {verdict}")

print("\nEvaluation complete.")

## Summary

| Concept | Where It's Used |
|---|---|
| **Structured prompting** | Each agent has a focused system instruction with explicit steps |
| **Function calling / tool use** | 5 tools registered; agents decide which to call and when |
| **Agentic pipeline** | `output_key` chains claim_analyzer → risk_scorer → report_writer |
| **Multi-agent system** | Orchestrator delegates to 3 specialized sub-agents |
| **Evaluation** | 4 labeled test cases with expected verdicts |

**Live demo:** https://rxhcc-app.vercel.app/  
**Full code:** https://github.com/sechan9999/rxhcc_agent

---
*Kaggle 5-Day AI Agents Intensive Vibe Coding — Agents for Good Track*